In [1]:
import os
from dotenv import load_dotenv

In [2]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print('No API key was found!')
elif not api_key.startswith('sk-proj-'):
    print('An API key was found, but it doesnt start with "sk-proj-", i.e, it is not an OpenAI API key')
elif api_key.strip() != api_key:
    print('An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them.')
else:
    print('The API key found and looks good so far!')


The API key found and looks good so far!


# Google ADK

The book's listing (pp. 187-188) is a technical support agent built around Human-in-the-Loop: it troubleshoots what it can, logs a ticket if the issue persists, and hands complex cases to a person through an `escalate_to_human` tool. A `personalization_callback` adds the customer's name, tier and purchases to every model request. As printed it never runs:

| The book | Here |
|---|---|
| `from google.adk.callbacks import CallbackContext` | no such module; `google.adk.agents.callback_context` (`CallbackContext` is now an alias of `Context`) |
| `from google.adk.models.llm import LlmRequest` | no such module; `google.adk.models` |
| `model="gemini-2.0-flash-exp"` | retired (`404 NOT_FOUND`); `gemini-flash-latest` |
| `personalization_callback` is defined but never attached | `before_model_callback=personalization_callback` on the agent |
| the callback inserts `Content(role="system")` into the conversation | Gemini rejects it: `400 INVALID_ARGUMENT. Role 'system' is not supported`. `llm_request.append_instructions([...])` adds the note to the system instruction instead |
| no runner, no customer in state | a session created with `state={"customer_info": ...}`, and two queries: one basic, one that should escalate |

The `system` role error is hidden in the book for the same reason as chapter 7's retired models: the callback was never attached, so it never ran. **Construction is not validation.**

Two things the listing still gets wrong, kept as written:

- The instruction says to check `state["customer_info"]["support_history"]`. As in chapter 12, ADK injects state only for `{key}` templates, and the callback's note leaves the history out, so the model never sees it. The customer below has a support history; watch whether any reply mentions it.
- The "human" is a placeholder tool that returns a success message. Nothing pauses and no person is asked. The agent *decides* to escalate; the loop back to a human is not built.

The code is split: the next cell holds the tools, the callback and the run helper, with no model. The Gemini cell and the Ollama cell each build the agent and run it, so either can run on its own.

In [3]:
# Book listing: Chapter 13, Hands-On Code Example (ADK), pp. 187-188.
# This cell holds the tools, the callback and a run helper, and no model. The Gemini cell
# and the Ollama cell below each build the agent and call run_support_queries.
import os
from typing import Optional

from google.adk.agents import Agent
from google.adk.agents.callback_context import CallbackContext  # book: google.adk.callbacks
from google.adk.models import LlmRequest  # book: google.adk.models.llm
from google.adk.runners import InMemoryRunner
from google.adk.tools.tool_context import ToolContext
from google.genai import types

# ADK 2.7.1 builds tool declarations with an experimental feature that is on by default,
# and warns once per process that it is. This switches it off; ADK falls back to its
# older function-declaration builder.
os.environ.setdefault("ADK_DISABLE_JSON_SCHEMA_FOR_FUNC_DECL", "1")


# Placeholder for tools (replace with actual implementations if needed)
# The prints are added so the run shows which tool the agent chose.
def troubleshoot_issue(issue: str) -> dict:
    print(f"  [tool] troubleshoot_issue({issue!r})")
    return {"status": "success", "report": f"Troubleshooting steps for {issue}."}

def create_ticket(issue_type: str, details: str) -> dict:
    print(f"  [tool] create_ticket({issue_type!r}, {details!r})")
    return {"status": "success", "ticket_id": "TICKET123"}

def escalate_to_human(issue_type: str) -> dict:
    # This would typically transfer to a human queue in a real system
    print(f"  [tool] escalate_to_human({issue_type!r})")
    return {"status": "success", "message": f"Escalated {issue_type} to a human specialist."}


def personalization_callback(
    callback_context: CallbackContext, llm_request: LlmRequest
) -> Optional[LlmRequest]:
    """Adds personalization information to the LLM request."""
    # Get customer info from state
    customer_info = callback_context.state.get("customer_info")
    if customer_info:
        customer_name = customer_info.get("name", "valued customer")
        customer_tier = customer_info.get("tier", "standard")
        recent_purchases = customer_info.get("recent_purchases", [])

        personalization_note = (
            f"\nIMPORTANT PERSONALIZATION:\n"
            f"Customer Name: {customer_name}\n"
            f"Customer Tier: {customer_tier}\n"
        )
        if recent_purchases:
            personalization_note += f"Recent Purchases: {', '.join(recent_purchases)}\n"

        # The book inserted Content(role="system") before the first content. Gemini
        # rejects that role (400 INVALID_ARGUMENT); the system instruction is where it goes.
        llm_request.append_instructions([personalization_note])
        print(f"  [callback] personalization added for {customer_name}")
    return None # Return None to continue with the modified request


# --- Run it: one customer, one basic issue, one that should go to a human ---
CUSTOMER_INFO = {
    "name": "Alex Chen",
    "tier": "premium",
    "recent_purchases": ["UltraBook Pro 14"],
    "support_history": ["2025-08: battery drained overnight, fixed by a BIOS update"],
}
QUERIES = [
    "My laptop's screen flickers. What should I do?",
    "I already tried everything: the laptop smells of burning plastic and the battery "
    "is swelling. I need someone to look at it.",
]


async def run_support_queries(agent) -> None:
    """Runs each query in a fresh session holding CUSTOMER_INFO and prints the replies."""
    runner = InMemoryRunner(agent=agent, app_name="support_app")
    for query in QUERIES:
        session = await runner.session_service.create_session(
            app_name="support_app", user_id="user_1", state={"customer_info": CUSTOMER_INFO}
        )
        print(f"\n>>> {query}")
        message = types.Content(role="user", parts=[types.Part(text=query)])
        async for event in runner.run_async(user_id="user_1", session_id=session.id, new_message=message):
            if event.content and event.content.parts:
                for part in event.content.parts:
                    if part.text and part.text.strip():
                        print(f"--- {event.author}: {part.text.strip()}")

In [4]:
# The book's agent on Gemini, with the callback attached.
from google.genai.types import GenerateContentConfig, AutomaticFunctionCallingConfig

technical_support_agent = Agent(
    name="technical_support_specialist",
    model="gemini-flash-latest",  # the book's gemini-2.0-flash-exp is retired
    instruction="""
You are a technical support specialist for our electronics company.
FIRST, check if the user has a support history in state["customer_info"]["support_history"]. If they do, reference this history in your responses.
For technical issues:
1. Use the troubleshoot_issue tool to analyze the problem.
2. Guide the user through basic troubleshooting steps.
3. If the issue persists, use create_ticket to log the issue.
For complex issues beyond basic troubleshooting:
1. Use escalate_to_human to transfer to a human specialist.
Maintain a professional but empathetic tone. Acknowledge the frustration technical issues can cause, while providing clear steps toward resolution.
""",
    tools=[troubleshoot_issue, create_ticket, escalate_to_human],
    # The book defines personalization_callback but never attaches it.
    before_model_callback=personalization_callback,
    # ADK runs its own tool loop and never uses the genai SDK's automatic function
    # calling, but leaves it enabled, so the SDK logs a one-time warning. This removes it.
    generate_content_config=GenerateContentConfig(
        automatic_function_calling=AutomaticFunctionCallingConfig(disable=True)
    ),
)

await run_support_queries(technical_support_agent)


>>> My laptop's screen flickers. What should I do?


  [callback] personalization added for Alex Chen


  [tool] troubleshoot_issue('screen flickering on UltraBook Pro 14')
  [callback] personalization added for Alex Chen


--- technical_support_specialist: Hello Alex,

I'm sorry to hear that you're experiencing screen flickering on your UltraBook Pro 14. Display issues can certainly be frustrating and disruptive to your work, but I'm here to help you get this sorted out.

Let's start by walking through a few basic troubleshooting steps to identify the cause:

1. **Test Task Manager:**
   - Press **Ctrl + Shift + Esc** to open **Task Manager**.
   - Watch the screen closely:
     - If **Task Manager flickers** along with the rest of the screen, the issue is likely related to your display driver or hardware.
     - If **everything except Task Manager flickers**, an incompatible app or software running in the background is likely causing the problem.

2. **Update or Roll Back Display Drivers:**
   - Right-click the **Start** button and select **Device Manager**.
   - Expand the **Display adapters** section.
   - Right-click your graphics card and select **Update driver** > **Search automatically for drivers

  [tool] escalate_to_human('battery swelling and burning smell')
  [callback] personalization added for Alex Chen


--- technical_support_specialist: Hello Alex,

I am so sorry to hear this, and I completely understand your concern. A swelling battery and the smell of burning plastic are serious hardware safety hazards, and I want to make sure you are safe.

**Immediate Safety Precautions:**
1. **Unplug the laptop immediately** from any power source or charger.
2. **Do not attempt to turn it on or charge it.**
3. If safe to do so, place the device in a well-ventilated, non-flammable area (away from carpets, fabrics, or flammable items).

Because this is a critical hardware issue beyond standard troubleshooting, **I have escalated your case directly to a human hardware specialist.** As a Premium customer, our specialist team will prioritize your UltraBook Pro 14 and reach out to you shortly to coordinate an inspection and resolution. 

Thank you for your patience, and please prioritize your safety in the meantime.


## Google ADK + Ollama

In [5]:
# Same support agent, running on a local model served by Ollama through LiteLLM.
# Prerequisites:
#   ollama pull qwen2.5:7b
#   litellm, from the google-adk[extensions] extra (uv sync, then restart the kernel)
# Needs only the shared cell above, not the Gemini cell, and no Google key.
#
# The instruction is shorter than the book's. With the book's wording a 7B model escalated
# the burning-battery case in 1 run of 3; in the other two it called no tool and asked the
# customer questions instead. Three plain rules, escalation first, escalated 3 of 3.
from google.adk.models.lite_llm import LiteLlm

os.environ.setdefault("OLLAMA_API_BASE", "http://localhost:11434")
OLLAMA_MODEL = "ollama_chat/qwen2.5:7b"

ollama_support_agent = Agent(
    name="technical_support_specialist",
    model=LiteLlm(model=OLLAMA_MODEL),
    instruction=(
        "You are a technical support specialist for an electronics company. "
        "If the issue is a safety hazard or beyond basic troubleshooting, call escalate_to_human. "
        "Otherwise call troubleshoot_issue, then guide the user through basic steps. "
        "If the issue persists, call create_ticket."
    ),
    tools=[troubleshoot_issue, create_ticket, escalate_to_human],
    before_model_callback=personalization_callback,
)

await run_support_queries(ollama_support_agent)


>>> My laptop's screen flickers. What should I do?
  [callback] personalization added for Alex Chen


  [tool] troubleshoot_issue('screen_flickers')
  [callback] personalization added for Alex Chen


--- technical_support_specialist: Let's try a few basic steps to resolve the screen flickering issue on your UltraBook Pro 14:

1. Ensure that your laptop is properly ventilated and not overheating. Check if there are any blocked air vents.
2. Update the graphics drivers to the latest version. You can find the latest drivers on our official website.
3. Run a power management tool to check if there are any issues with power settings that might be causing the flickering.
4. If you recently installed any new software or hardware, try disabling or removing them to see if the issue persists.
5. Check for any software updates and install them if available.

Please follow these steps and let me know if the issue persists after attempting these solutions.

>>> I already tried everything: the laptop smells of burning plastic and the battery is swelling. I need someone to look at it.
  [callback] personalization added for Alex Chen


  [tool] escalate_to_human('Severe hardware issue with premium device')
  [callback] personalization added for Alex Chen


--- technical_support_specialist: I've escalated your issue to a human specialist who will be able to assist you further. Please provide them with your UltraBook Pro 14 serial number for a quicker resolution.
